<a href="https://colab.research.google.com/github/Sapikzzz/AI-ASL/blob/main/Transer_with_zero_weights.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
grassknoted_asl_alphabet_path = kagglehub.dataset_download('grassknoted/asl-alphabet')

print('Data source import complete.')


Data source import complete.


In [2]:
import numpy as np
import os
import cv2
from sklearn.utils import shuffle
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import gc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization
import kagglehub
import seaborn as sns
import pandas as pd
import skimage
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
import os
from glob import glob
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [3]:
# Download latest version
grassknoted_asl_alphabet_path = kagglehub.dataset_download("grassknoted/asl-alphabet")
print(grassknoted_asl_alphabet_path)
train_path = os.path.join(grassknoted_asl_alphabet_path, '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train')

/kaggle/input/asl-alphabet


In [4]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices()

tf.test.is_gpu_available()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            print("GPU configured successfully")
    except RuntimeError as e:
        print("Error configuring GPU:", e)

Num GPUs Available:  1


Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


GPU configured successfully


In [5]:
target_size = (96, 96)
target_dims = (96, 96, 3)
n_classes = 29
val_frac = 0.1
batch_size = 256

data_augmentor = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
    rescale=1./255,
    validation_split=val_frac
)

In [6]:
train_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, shuffle=True, subset='training')
val_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, subset='validation')

Found 78300 images belonging to 29 classes.
Found 8700 images belonging to 29 classes.


In [7]:
MobileNet_layers = MobileNetV2(weights='imagenet', include_top=False, input_shape=target_dims)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [8]:
model = Sequential([
    MobileNet_layers,
    GlobalAveragePooling2D(),
    Dense(1024, activation='relu'),
    Dropout(0.5),
    Dense(n_classes, activation='softmax')
])

In [9]:
for layer in MobileNet_layers.layers:
    if hasattr(layer, 'kernel_initializer'):
        layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))

In [10]:
for layer in MobileNet_layers.layers:
    layer.trainable = True

In [11]:
model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

In [12]:
MobileNet_history = model.fit(train_gen, epochs=5, validation_data=val_gen)

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 743s 2s/step - accuracy: 0.4710 - loss: 1.8228 - val_accuracy: 0.0492 - val_loss: 11.3112
Epoch 2/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 305s 998ms/step - accuracy: 0.9606 - loss: 0.1236 - val_accuracy: 0.0345 - val_loss: 14.3080
Epoch 3/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 310s 1s/step - accuracy: 0.9819 - loss: 0.0584 - val_accuracy: 0.0345 - val_loss: 19.6162
Epoch 4/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 307s 1s/step - accuracy: 0.9865 - loss: 0.0431 - val_accuracy: 0.0345 - val_loss: 24.8222
Epoch 5/5
306/306 ━━━━━━━━━━━━━━━━━━━━ 309s 1s/step - accuracy: 0.9883 - loss: 0.0394 - val_accuracy: 0.0517 - val_loss: 18.0047


In [13]:
model.save('/MobileNet_model.keras')